# 00 Estimate Linear Lambda Init

Runs tuned LinearBidder on the train split and estimates the constant DRLB lambda init as `ctr_pred / bid`.

In [1]:
import sys
import pickle
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.linear_lambda import estimate_linear_lambda_init
from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.infra.split_utils import resolve_normalized_splits


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Tuned linear params live under evaluate_baselines/best_params/<subfolder>/ (see baselines_finetune.BaseLineTrainer).
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
# Last expression must be at module level - Jupyter does not auto-display values inside with / if / etc.
linear_tuned_params


{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [3]:
config_ref = build_drlb_config(
    run_name='may06_estimate_linear_lambda_init',
    profile='may05_default_best_fixed',
    split_set='full_train_val_holdout',
)
normalized_splits = resolve_normalized_splits(config_ref)
linear_params = {
    'cold_start_coef': linear_tuned_params['coef'],
    'lower_clip': linear_tuned_params['lower_clip'],
    'upper_clip': linear_tuned_params['upper_clip'],
    'factor': linear_tuned_params['factor'],
}
linear_params


{'cold_start_coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [4]:
linear_lambda_info = estimate_linear_lambda_init(
    normalized_splits=normalized_splits,
    linear_params=linear_params,
    auction_mode=config_ref.auction_mode,
    split_roles=["train", "val"],
)
linear_lambda_init = linear_lambda_info['linear_lambda_init']
linear_lambda_median = linear_lambda_info['linear_lambda_median']
linear_lambda_summary = linear_lambda_info['linear_lambda_summary']
linear_lambda_init, linear_lambda_median, linear_lambda_summary


(0.0033706948894393117,
 0.0009182510525206816,
 count    5.259100e+04
 mean     3.370695e-03
 std      1.316247e-02
 min      8.457360e-07
 10%      1.055850e-04
 25%      3.008268e-04
 50%      9.182511e-04
 75%      2.653919e-03
 90%      7.389486e-03
 max      1.218054e+00
 Name: linear_lambda, dtype: float64)

In [5]:
linear_lambda_info['linear_lambda_examples']


,campaign_id,prev_timestamp,bid,ctr_pred,linear_lambda
0,2978176,529650000,8.960722,0.031135,0.003475
1,2978176,529653600,10.699321,0.015867,0.001483
2,2978176,529657200,12.839185,0.024953,0.001944
3,2978176,529660800,15.407022,0.037718,0.002448
4,2978176,529664400,18.488426,0.024103,0.001304
5,2978176,529668000,22.186111,0.017334,0.000781
6,2978176,529671600,26.623333,0.048282,0.001814
7,2978176,529675200,31.948000,0.033528,0.001049
8,2978176,529678800,38.337600,0.041229,0.001075
9,2978176,529682400,46.005120,0.015753,0.000342
